# 55 Input & Result (100% datos) — Estimador de Precio

Igual que `55_input_result` pero entrenando sobre **todas las observaciones** (sin split train/test).

| Modelo | Datos | CV-RMSE | Test R² |
|---|---|---|---|
| XGBoost Sale (`53_boost_sale_optuna`) | `final_sale_idealistaAPI.csv` | 0.233 | ≈ 0.85 |
| XGBoost Rent (`53_boost_rent`) | `final_rent_idealistaAPI.csv` | 0.146 | ≈ 0.62 |

**Por qué 100%:** el split 80/20 sirve para evaluar generalización. Una vez aprobadas las métricas en `55_sale_rent_models`, re-entrenar con todos los datos es la práctica estándar — el modelo ve más ejemplos y es ligeramente mejor.

**Intervalo de error:** se usa el **CV-RMSE** (5-fold) obtenido en `55_sale_rent_models`, más robusto que el RMSE de un único split 20%.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)

# ── Configuración ──────────────────────────────────────────────────────────────
RANDOM_STATE              = 42
TEST_SIZE                 = 0.20
TARGET_COL                = "log_precio"
IQR_FACTOR                = 1.5
MIN_MUNI_OBS              = 10
SALE_PRECIO_M2_MIN        = 1000   # €/m²        — suelo venta
RENT_PRECIO_M2_VACACIONAL = 18.0   # €/m²/mes   — techo alquiler
RENT_PRECIO_M2_MIN        = 6.0    # €/m²/mes   — suelo alquiler

# ── Rutas ──────────────────────────────────────────────────────────────────────
def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "gold").exists():
            return p
    raise FileNotFoundError("No se encontró la raíz del proyecto (data/gold)")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SALE_PATH    = PROJECT_ROOT / "data" / "gold" / "final_sale_idealistaAPI.csv"
RENT_PATH    = PROJECT_ROOT / "data" / "gold" / "final_rent_idealistaAPI.csv"

print(f"Proyecto: {PROJECT_ROOT}")

In [ ]:
# ── Features sale (Optuna: incluye latitud, longitud, es_exterior_piso, tiene_ascensor_piso) ──
SALE_BASE_FEATURES = [
    "superficie_construida_m2",
    "numero_dormitorios",
    "numero_banos",
    "latitud",
    "longitud",
    "planta_num",
    "es_exterior_piso",
    "tiene_ascensor_piso",
    "tiene_garaje",
    "obra_nueva",
    "distancia_min_playa_km",
    "distancia_min_supermercado_km",
    "distancia_min_colegio_km",
    "precio_m2_municipio_media",
    "ratio_dormitorios_superficie",
    "ratio_banos_superficie",
    "interaccion_planta_sin_ascensor_piso",
    "distancia_centro_municipio_km",
    "score_cercania_servicios",
    "tipologia_unificada_piso",
    "tipologia_unificada_unifamiliar",
]

# ── Features rent (incluye es_exterior_piso, tiene_ascensor_piso) ─────────────
RENT_BASE_FEATURES = [
    "superficie_construida_m2",
    "numero_dormitorios",
    "numero_banos",
    "planta_num",
    "es_exterior_piso",
    "tiene_ascensor_piso",
    "tiene_garaje",
    "obra_nueva",
    "distancia_min_playa_km",
    "distancia_min_supermercado_km",
    "distancia_min_colegio_km",
    "precio_m2_municipio_media",
    "ratio_dormitorios_superficie",
    "ratio_banos_superficie",
    "interaccion_planta_sin_ascensor_piso",
    "distancia_centro_municipio_km",
    "score_cercania_servicios",
    "tipologia_unificada_piso",
    "tipologia_unificada_unifamiliar",
]

# ── Hiperparámetros (idénticos a 55_sale_rent_models) ─────────────────────────
# Sale: Optuna (53_boost_sale_optuna, trial #76, CV-RMSE=0.23347, test R²=0.853)
XGB_PARAMS_SALE = dict(
    n_estimators=950,
    max_depth=6,
    learning_rate=0.026390709496515886,
    subsample=0.6705806157309522,
    colsample_bytree=0.7312116009128224,
    min_child_weight=9,
    reg_lambda=1.6752171349993321,
    reg_alpha=0.3505729220414384,
    gamma=0.005208850498085864,
    random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
)

XGB_PARAMS_RENT = dict(
    n_estimators=1000, max_depth=6, learning_rate=0.011737724486287017,
    subsample=0.6046283826235364, colsample_bytree=0.8531891359517659,
    min_child_weight=6, reg_lambda=3.378990393436524,
    reg_alpha=0.06467616791725625, gamma=0.03464694507245766,
    random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
)

print("Configuración cargada.")


In [ ]:
def build_X(df: pd.DataFrame, base_features: list) -> tuple:
    """Construye X, feature_cols y medianas de entrenamiento para imputación."""
    df2 = df.copy()
    base = [f for f in base_features if f in df2.columns]

    mun_cols = sorted([c for c in df2.columns if c.startswith("municipio_")])
    if mun_cols:
        counts = df2[mun_cols].sum()
        small  = counts[counts < MIN_MUNI_OBS].index.tolist()
        if small:
            df2["municipio_otros"] = df2[small].max(axis=1)
            df2 = df2.drop(columns=small)
        mun_final = sorted(c for c in df2.columns if c.startswith("municipio_"))
    else:
        mun_final = []

    all_feats = base + [m for m in mun_final if m not in base]
    X_raw = df2[all_feats].copy()
    medians = X_raw.median().to_dict()          # guardamos antes de imputar
    imputer = SimpleImputer(strategy="median")
    X = pd.DataFrame(imputer.fit_transform(X_raw), columns=X_raw.columns, index=X_raw.index)
    return X, all_feats, medians

In [ ]:
# CV-RMSE de 55_sale_rent_models (5-fold sobre train completo)
# Sale: 53_boost_sale_optuna (Optuna, trial #76)
SALE_CV_RMSE = 0.23347
RENT_CV_RMSE = 0.14622


In [ ]:
GEO_COLS = [
    "distancia_min_playa_km",
    "distancia_min_supermercado_km",
    "distancia_min_colegio_km",
    "precio_m2_municipio_media",
    "distancia_centro_municipio_km",
    "score_cercania_servicios",
    "latitud",
    "longitud",
]

def build_geo_ref(df: pd.DataFrame) -> pd.DataFrame:
    """Mediana de variables geográficas por municipio a partir del dataset limpio."""
    mun_cols = [c for c in df.columns if c.startswith("municipio_")]
    rows = []
    for mc in mun_cols:
        nombre = mc.replace("municipio_", "")
        subset = df[df[mc] == 1]
        if len(subset) == 0:
            continue
        row = {"municipio": nombre}
        for gc in GEO_COLS:
            row[gc] = subset[gc].median() if gc in subset.columns else np.nan
        rows.append(row)
    return pd.DataFrame(rows).set_index("municipio")

sale_geo_ref = build_geo_ref(df_sale)
rent_geo_ref = build_geo_ref(df_rent)

print(f"Referencia geográfica — Sale: {len(sale_geo_ref)} municipios")
print(f"Referencia geográfica — Rent: {len(rent_geo_ref)} municipios")


In [ ]:
print("Municipios disponibles (VENTA):")
print(sorted(sale_geo_ref.index.tolist()))
print()
print("Municipios disponibles (ALQUILER):")
print(sorted(rent_geo_ref.index.tolist()))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  ESTIMADOR DE PRECIO — modifica los valores y ejecuta la celda
# ══════════════════════════════════════════════════════════════════════════════

# ── Datos de la vivienda ──────────────────────────────────────────────────────
MUNICIPIO      = "Santander"   # Nombre exacto (ver celda anterior)
SUPERFICIE_M2  = 200            # m² construidos
N_DORMITORIOS  = 5             # Número de dormitorios
N_BANOS        = 3             # Número de baños
TIENE_GARAJE   = True         # True / False
OBRA_NUEVA     = False         # True / False
TIPOLOGIA      = "unifamiliar"        # "piso"  o  "unifamiliar"

# ── Solo para PISO (pon None si es unifamiliar) ───────────────────────────────
PLANTA         = 5             # 0 = bajo, 1 = primera…  → None si unifamiliar
ES_EXTERIOR    = True          # True / False             → None si unifamiliar
TIENE_ASCENSOR = True          # True / False             → None si unifamiliar


# ── Lógica interna ────────────────────────────────────────────────────────────
def _build_row(municipio, sup, dorm, banos, planta, exterior, ascensor,
               garaje, obra_nueva, tipologia, feature_cols, geo_ref, medians):
    row = pd.Series(np.nan, index=feature_cols)

    def _set(k, v):
        if k in row.index and v is not None:
            row[k] = float(v)

    _set("superficie_construida_m2",        sup)
    _set("numero_dormitorios",              dorm)
    _set("numero_banos",                    banos)
    _set("tiene_garaje",                    int(garaje))
    _set("obra_nueva",                      int(obra_nueva))
    _set("tipologia_unificada_piso",        1 if tipologia == "piso"        else 0)
    _set("tipologia_unificada_unifamiliar",  1 if tipologia == "unifamiliar" else 0)
    _set("planta_num",                      planta)
    _set("es_exterior_piso",               int(exterior) if exterior is not None else None)
    _set("tiene_ascensor_piso",            int(ascensor) if ascensor is not None else None)

    if tipologia == "piso" and planta is not None and ascensor is not None:
        _set("interaccion_planta_sin_ascensor_piso", planta * (1 - int(ascensor)))
    else:
        _set("interaccion_planta_sin_ascensor_piso", 0)

    _set("ratio_dormitorios_superficie", dorm / sup if sup > 0 else 0)
    _set("ratio_banos_superficie",       banos / sup if sup > 0 else 0)

    if municipio in geo_ref.index:
        for col, val in geo_ref.loc[municipio].to_dict().items():
            _set(col, val)

    for c in [c for c in feature_cols if c.startswith("municipio_")]:
        row[c] = 0.0
    mun_col = f"municipio_{municipio}"
    if mun_col in row.index:
        row[mun_col] = 1.0
    elif "municipio_otros" in row.index:
        row["municipio_otros"] = 1.0

    for col in feature_cols:
        if pd.isna(row[col]):
            row[col] = medians.get(col, 0)

    return pd.DataFrame([row])


def _rango(precio: float, rmse_log: float) -> tuple:
    """Intervalo ±1 RMSE en espacio log → rango asimétrico en precio real."""
    return precio * np.exp(-rmse_log), precio * np.exp(rmse_log)


def estimar_precio(municipio, sup, dorm, banos, planta, exterior,
                   ascensor, garaje, obra_nueva, tipologia):
    desc = []
    if tipologia == "piso":
        if planta   is not None: desc.append(f"Planta {planta}")
        if exterior is not None: desc.append("exterior" if exterior else "interior")
        if ascensor is not None: desc.append("con ascensor" if ascensor else "sin ascensor")
    if garaje:     desc.append("garaje")
    if obra_nueva: desc.append("obra nueva")

    print()
    print("\u2550" * 58)
    print(f"  {sup} m\u00b2  \u00b7  {dorm} dorm.  \u00b7  {banos} ba\u00f1os  \u2014  {municipio}")
    print(f"  {tipologia.upper()}  \u00b7  {' \u00b7 '.join(desc) if desc else '\u2014'}")
    print("\u2550" * 58)

    precio_venta    = None
    precio_alquiler = None

    # ── VENTA ──────────────────────────────────────────────────────────────────
    if municipio not in sale_geo_ref.index:
        print(f"\n  \u26a0  '{municipio}' no disponible en venta")
    else:
        X_pred = _build_row(municipio, sup, dorm, banos, planta, exterior,
                            ascensor, garaje, obra_nueva, tipologia,
                            feats_sale, sale_geo_ref, medians_sale)
        precio_venta = np.exp(model_sale.predict(X_pred)[0])
        v_lo, v_hi   = _rango(precio_venta, SALE_CV_RMSE)
        pct_sale     = (np.exp(SALE_CV_RMSE) - 1) * 100
        print(f"\n  Precio de venta estimado   : {precio_venta:>12,.0f} \u20ac")
        print(f"  Intervalo error (\u00b11\u03c3)      : [{v_lo:>10,.0f} \u20ac  \u2014  {v_hi:>10,.0f} \u20ac]  (\u00b1{pct_sale:.0f}%)")

    # ── ALQUILER ───────────────────────────────────────────────────────────────
    if municipio not in rent_geo_ref.index:
        print(f"  \u26a0  '{municipio}' no disponible en alquiler")
    else:
        X_pred = _build_row(municipio, sup, dorm, banos, planta, exterior,
                            ascensor, garaje, obra_nueva, tipologia,
                            feats_rent, rent_geo_ref, medians_rent)
        precio_alquiler = np.exp(model_rent.predict(X_pred)[0])
        r_lo, r_hi      = _rango(precio_alquiler, RENT_CV_RMSE)
        pct_rent        = (np.exp(RENT_CV_RMSE) - 1) * 100
        print(f"\n  Alquiler mensual estimado  : {precio_alquiler:>12,.0f} \u20ac/mes")
        print(f"  Intervalo error (\u00b11\u03c3)      : [{r_lo:>10,.0f} \u20ac/mes  \u2014  {r_hi:>10,.0f} \u20ac/mes]  (\u00b1{pct_rent:.0f}%)")

    # ── RENTABILIDAD ───────────────────────────────────────────────────────────
    if precio_venta and precio_alquiler:
        rentabilidad = (precio_alquiler * 12) / precio_venta * 100
        print(f"\n  Rentabilidad bruta estim.  : {rentabilidad:>11.1f} %")

    print()


# ── Ejecutar estimación ───────────────────────────────────────────────────────
estimar_precio(
    MUNICIPIO, SUPERFICIE_M2, N_DORMITORIOS, N_BANOS,
    PLANTA, ES_EXTERIOR, TIENE_ASCENSOR, TIENE_GARAJE, OBRA_NUEVA, TIPOLOGIA
)